# Universal Document Fine-Tuner (Unsloth + QLoRA): cleaned version

Ingests a PDF / DOCX / TXT document, uses a **teacher** model to generate synthetic Q&A, fine-tunes a small
**student** model, checks it on held-out questions, and exports GGUF.

**What was fixed compared to the previous run** (where every answer was *"does not contain this information"*):

| problem | fix |
|---|---|
| 24 examples, 4 of them the *same* refusal, so the model learned to always refuse | ~5 Q&A per chunk **+ 3 paraphrases each**; refusals capped at ~12% of the data, with varied questions |
| System prompt said "use the provided context", but no context was ever given | Prompt matches reality; ~30% of examples include retrieved excerpts, so both modes are trained |
| Hallucination filter kept anything sharing one word with the chunk | Numbers and named terms must appear in the chunk, and most answer words must come from it |
| Out-of-order cells, old pinned installs, `gguf_files` undefined | One install cell; every cell depends only on cells above it. **Use Runtime -> Run all.** |
| Deprecated `SFTTrainer(tokenizer=..., dataset_text_field=...)` | Plain `transformers.Trainer` with answer-only labels (`-100` on the prompt) |
| No test at all | Held-out paraphrases + held-out refusals, scored automatically |

## 1. Install (run once, then *Run all*)

In [ ]:
!pip install -q unsloth pypdf python-docx scikit-learn

## 2. Configuration

In [ ]:
DOC_PATH      = "/content/your_document.docx"      # .pdf / .docx / .txt / .md, upload via the left sidebar
TEACHER_MODEL = "unsloth/Qwen2.5-3B-Instruct"      # writes the Q&A ("unsloth/Qwen2.5-1.5B-Instruct" = faster)
STUDENT_MODEL = "unsloth/Qwen2.5-0.5B-Instruct"    # model that is fine-tuned and exported
MAX_SEQ_LENGTH = 4096
QA_PER_CHUNK, N_PARAPHRASES = 5, 3
REFUSAL_SHARE  = 0.12          # refusals as a share of training examples (too many -> model refuses everything)
CONTEXT_SHARE  = 0.3           # share of training examples that include retrieved excerpts
EPOCHS, LR     = 5, 2e-4
MAX_STEPS      = -1            # e.g. 10 for a smoke test
SEED           = 3407
REFUSAL = "The provided document does not contain this information."

## 3. Imports and helpers

In [ ]:
import os, re, json, random, gc, glob
import numpy as np, torch
from unsloth import FastLanguageModel           # must be imported before transformers
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from sklearn.feature_extraction.text import TfidfVectorizer

random.seed(SEED); rng = random.Random(SEED)

def load(name):
    m, tok = FastLanguageModel.from_pretrained(model_name=name, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return m, tok

@torch.no_grad()
def generate(m, tok, conversations, max_new_tokens=256, batch_size=8):
    # batched greedy generation using the tokenizer's own chat template; returns only the new text
    FastLanguageModel.for_inference(m)
    tok.padding_side = "left"
    out_texts = []
    for i in range(0, len(conversations), batch_size):
        prompts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversations[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05,
                         pad_token_id=tok.pad_token_id)
        out_texts += [tok.decode(r, skip_special_tokens=True).strip() for r in out[:, enc["input_ids"].shape[1]:]]
    return out_texts

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

## 4. Document ingestion and sliding-window chunking (400 words, 50 overlap)

In [ ]:
def read_document(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}. Upload it via the Colab sidebar and fix DOC_PATH.")
    head = open(path, "rb").read(4)
    if head == b"%PDF":
        from pypdf import PdfReader
        return "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
    if head == b"PK\x03\x04":                          # .docx: paragraphs AND tables, in document order
        from docx import Document
        from docx.table import Table
        from docx.text.paragraph import Paragraph
        doc = Document(path)
        def walk(parent, el):
            for child in el.iterchildren():
                tag = child.tag.split("}")[-1]
                if tag == "p":
                    yield Paragraph(child, parent).text
                elif tag == "tbl":
                    seen = set()
                    for row in Table(child, parent).rows:
                        for cell in row.cells:
                            if cell._tc not in seen:   # merged cells repeat
                                seen.add(cell._tc); yield from walk(cell, cell._tc)
        return "\n".join(t for t in walk(doc, doc.element.body) if t.strip())
    return open(path, encoding="utf-8", errors="replace").read()

text = read_document(DOC_PATH)
DOC_TITLE = os.path.splitext(os.path.basename(DOC_PATH))[0].replace("_", " ")
words = text.split()
chunks = [" ".join(words[i:i + 400]) for i in range(0, max(1, len(words) - 50), 350)]
print(f"'{DOC_TITLE}': {len(words)} words -> {len(chunks)} chunks")

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english").fit(chunks)
chunk_vecs = tfidf.transform(chunks)
def retrieve(q, k=2):
    sims = (chunk_vecs @ tfidf.transform([q]).T).toarray().ravel()
    return [(chunks[i], float(sims[i])) for i in np.argsort(-sims)[:k]]
def context_for(q, k=2):
    return "\n\n---\n\n".join(c for c, _ in retrieve(q, k))

## 5. Synthetic Q&A with the teacher (batched) and a real hallucination filter

In [ ]:
PROMPT = '''Write {n} diverse question-answer pairs answered ONLY by the excerpt below
(facts with names/numbers/dates, definitions, and "how/what steps" questions).
Answers: 1-3 sentences, copy names and numbers exactly, no outside knowledge.
For each question add {p} paraphrases (same meaning, different wording).
Return ONLY JSON: [{{"question": "...", "paraphrases": ["...", "...", "..."], "answer": "..."}}]

<excerpt>
{chunk}
</excerpt>'''

def parse_items(raw):
    m = re.search(r"\[.*\]", raw, re.S)
    try:
        return [d for d in json.loads(m.group(0)) if isinstance(d, dict)] if m else []
    except json.JSONDecodeError:                   # salvage individual objects
        out = []
        for obj in re.findall(r"\{[^{}]*\}", raw, re.S):
            try: out.append(json.loads(obj))
            except json.JSONDecodeError: pass
        return out

def norm(s): return re.sub(r"\s+", " ", s.lower().replace("’", "'").replace("–", "-").replace("—", "-"))
STOP = set("a an the and or of to in on for with by at from as is are was were be this that it its they you we if when what which who how why".split())
def numbers(t): return {n.rstrip(".,") for n in re.findall(r"\d[\d.,:/%-]*", t)}
def entities(t):
    t = re.sub(r"(^|[.!?:]\s+)([A-Z])(?=[a-z])", lambda m: m.group(1) + m.group(2).lower(), t)   # ignore sentence starts
    return {e for e in re.findall(r"\b[A-Z][\w&-]*(?:\s+[A-Z][\w&-]*)*\b", t) if e.lower() not in STOP}

def grounded(answer, chunk):
    src = norm(chunk)
    if any(n not in src for n in numbers(answer)) or any(norm(e) not in src for e in entities(answer)):
        return False
    cw = [w for w in re.findall(r"[a-z0-9]+", norm(answer)) if len(w) > 3 and w not in STOP]
    return not cw or sum(w in src for w in cw) / len(cw) >= 0.6

teacher, t_tok = load(TEACHER_MODEL)
raws = generate(teacher, t_tok, [[{"role": "user", "content": PROMPT.format(n=QA_PER_CHUNK, p=N_PARAPHRASES, chunk=c)}]
                                 for c in chunks], max_new_tokens=1000, batch_size=4)

facts, seen, dropped = [], set(), 0
for chunk, raw in zip(chunks, raws):
    for it in parse_items(raw):
        q, a = str(it.get("question", "")).strip(), str(it.get("answer", "")).strip()
        if not q or not a or norm(q) in seen or not grounded(a, chunk):
            dropped += 1; continue
        qs = [q] + [p.strip() for p in it.get("paraphrases", []) if isinstance(p, str) and p.strip() and norm(p) not in seen]
        seen.update(norm(x) for x in qs)
        facts.append({"questions": qs, "answer": a})
print(f"{len(facts)} grounded facts ({sum(len(f['questions']) for f in facts)} questions), {dropped} pairs dropped")
assert len(facts) >= 5, "Too few facts - check the extracted text or use a larger teacher"
del teacher, t_tok; free_gpu()

## 6. Refusal data (balanced), train/test split
Refusals are capped at ~12% of the data, so "I don't know" is never the model's safest answer.
Test set: one held-out **paraphrase** of every fact (wording never trained), plus held-out refusal questions.

In [ ]:
OUT_OF_SCOPE = [
    "What is the capital of France?", "Who painted the Mona Lisa?", "How do I bake a chocolate cake?",
    "What is the meaning of life?", "How many planets are in the solar system?", "Who won the 2018 football World Cup?",
    "What is the boiling point of water in Fahrenheit?", "Translate 'good morning' into Spanish.",
    "What is the speed of light?", "Who wrote Hamlet?", "How do I change a car tyre?", "What is photosynthesis?",
    "Recommend a good movie for tonight.", "What is the tallest mountain in the world?", "How do I invest in stocks?",
    "What is the population of Japan?", "Who invented the telephone?", "What is the chemical symbol for gold?",
    "How many bones are in the human body?", "What is the weather tomorrow?",
]
oos = [q for q in OUT_OF_SCOPE if retrieve(q, 1)[0][1] < 0.12]      # drop anything the document might answer
rng.shuffle(oos)

train, test = [], []
for f in facts:
    qs = f["questions"][:]
    if len(qs) >= 2:
        test.append({"q": qs.pop(), "a": f["answer"], "kind": "paraphrase"})
    train += [{"q": q, "a": f["answer"]} for q in qs]
n_ref = max(4, round(REFUSAL_SHARE * len(train)))
test += [{"q": q, "a": REFUSAL, "kind": "refusal"} for q in oos[:5]]
train += [{"q": q, "a": REFUSAL} for q in (oos[5:] * 3)[:n_ref]]
rng.shuffle(train)
print(f"train={len(train)} (refusals={n_ref}), test={len(test)}")
for r in train[:3]: print(r)

## 7. Student model: answer-only loss masking + QLoRA training

In [ ]:
SYSTEM = (f"You are an assistant for the document '{DOC_TITLE}'. Answer accurately and concisely from the document. "
          f"If the document does not contain the answer, reply exactly: '{REFUSAL}'")

def messages(q, context=None):
    sys_msg = SYSTEM + (f"\n\nDocument excerpts:\n{context}" if context else "")
    return [{"role": "system", "content": sys_msg}, {"role": "user", "content": q}]

model, tokenizer = load(STUDENT_MODEL)

def encode(row):
    msgs = messages(row["q"], context_for(row["q"]) if rng.random() < CONTEXT_SHARE else None)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(msgs + [{"role": "assistant", "content": row["a"]}], tokenize=False)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids),
            "labels": [-100] * len(p_ids) + f_ids[len(p_ids):]}      # loss only on the answer (+ end token)

train_ds = Dataset.from_list(train).map(encode, remove_columns=["q", "a"])

model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                         use_gradient_checkpointing="unsloth", random_state=SEED)
FastLanguageModel.for_training(model)
tokenizer.padding_side = "right"
trainer = Trainer(
    model=model, train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    args=TrainingArguments(output_dir="outputs", num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
                           per_device_train_batch_size=8, gradient_accumulation_steps=1, learning_rate=LR,
                           lr_scheduler_type="cosine", warmup_steps=5,
                           fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
                           optim="adamw_8bit", logging_steps=10, save_strategy="no", report_to="none", seed=SEED))
trainer.train()     # healthy: loss ends well below 0.5

## 8. Test on held-out questions
* **paraphrase**: trained facts asked in new wording. Shows whether the facts were learned.
* **refusal**: unseen off-topic questions. These should be refused, while document questions should not be.

In [ ]:
def f1(pred, ref):
    p, r = re.findall(r"[a-z0-9]+", norm(pred)), re.findall(r"[a-z0-9]+", norm(ref))
    common = sum(min(p.count(w), r.count(w)) for w in set(p))
    return 0.0 if not common else 2 * common / (len(p) + len(r))
is_refusal = lambda t: "does not contain this information" in norm(t)

for mode in ["fine-tuned", "fine-tuned + retrieved excerpts"]:
    preds = generate(model, tokenizer, [messages(r["q"], context_for(r["q"]) if "retrieved" in mode else None) for r in test],
                     max_new_tokens=150)
    para = [(r, p) for r, p in zip(test, preds) if r["kind"] == "paraphrase"]
    ref = [(r, p) for r, p in zip(test, preds) if r["kind"] == "refusal"]
    print(f"\n=== {mode} ===")
    print(f"paraphrase F1: {np.mean([f1(p, r['a']) for r, p in para]):.2f} | "
          f"false refusals: {np.mean([is_refusal(p) for _, p in para]):.0%} | "
          f"refusal accuracy: {np.mean([is_refusal(p) for _, p in ref]):.0%}")
    for r, p in para[:3]:
        print(f"Q: {r['q']}\n   expected: {r['a'][:140]}\n   model   : {p[:140]}")

## 9. Export GGUF (q4_k_m) and the Ollama Modelfile

In [ ]:
model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
model.save_pretrained_gguf("model_export", tokenizer, quantization_method="q4_k_m")
gguf_files = sorted(glob.glob("model_export*/**/*.gguf", recursive=True) + glob.glob("model_export*/*.gguf"))
gguf_files = [f for f in gguf_files if "q4_k_m" in f.lower()] or gguf_files
assert gguf_files, "GGUF export produced no file"
GGUF_PATH = gguf_files[0]
open("system_prompt.txt", "w").write(SYSTEM)
print("GGUF:", GGUF_PATH, f"({os.path.getsize(GGUF_PATH) / 1e6:.0f} MB)")

## 10. Ask the GGUF model (llama-cpp-python, CPU)
`create_chat_completion` uses the chat template stored inside the GGUF file. `grounded=True` adds retrieved excerpts, which gives the most accurate answers for a 0.5B model.

In [ ]:
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
from llama_cpp import Llama
GGUF_PATH = sorted(glob.glob("model_export*/**/*.gguf", recursive=True) + glob.glob("model_export*/*.gguf"),
                   key=lambda f: "q4_k_m" not in f.lower())[0]      # self-contained: no hidden state
llm = Llama(model_path=GGUF_PATH, n_ctx=MAX_SEQ_LENGTH, n_threads=os.cpu_count(), verbose=False)

def ask(question, grounded=True):
    msgs = messages(question, context_for(question) if grounded else None)
    return llm.create_chat_completion(messages=msgs, temperature=0.0, max_tokens=200)["choices"][0]["message"]["content"].strip()

for r in test[:3] + test[-1:]:
    print("Q:", r["q"]); print("A:", ask(r["q"])); print("   expected:", r["a"][:140], "\n")